# ST-GCN Joint — independent evaluation and error analysis

This notebook freezes the best logged checkpoint, independently runs the complete NTU60 `xsub_val` split, exports predictions, and creates all confusion, confidence, training-curve, and README artifacts. It does not train any model.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/mzuyyy/Human-action-recognition.git'
PROJECT_DIR = Path('/kaggle/working/ntu-action-recognition')
MMACTION2_DIR = Path('/kaggle/working/mmaction2')
ORIGINAL_WORK_DIR = PROJECT_DIR / 'work_dirs/stgcn_ntu60_xsub_40e'
RESUME_WORK_DIR = PROJECT_DIR / 'work_dirs/stgcn_ntu60_xsub_80e_resume'
ANN_FILE = PROJECT_DIR / 'data/skeleton/ntu60_2d.pkl'
NTU60_URL = 'https://download.openmmlab.com/mmaction/v1.0/skeleton/data/ntu60_2d.pkl'

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print('project:', PROJECT_DIR)
ORIGINAL_WORK_DIR.mkdir(parents=True, exist_ok=True)
RESUME_WORK_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
%%bash
set -euo pipefail
python -m pip uninstall -q -y mmcv mmcv-lite >/dev/null 2>&1 || true
python -m pip install -q --only-binary=mmcv-lite \
  "importlib-metadata" "mmengine>=0.7.1,<1.0.0" "mmcv-lite==2.1.0"

MMACTION2_SRC=/kaggle/working/mmaction2
if [ ! -d "${MMACTION2_SRC}/.git" ]; then
  git -c advice.detachedHead=false clone --branch v1.2.0 --depth 1 \
    https://github.com/open-mmlab/mmaction2.git "${MMACTION2_SRC}"
else
  if ! git -C "${MMACTION2_SRC}" rev-parse -q --verify \
      "refs/tags/v1.2.0^{commit}" >/dev/null; then
    git -C "${MMACTION2_SRC}" fetch -q --depth 1 origin tag v1.2.0
  fi
  git -c advice.detachedHead=false -C "${MMACTION2_SRC}" \
    checkout -q --detach v1.2.0
fi
python -m pip uninstall -q -y mmaction2 >/dev/null 2>&1 || true
python -m pip install -q -e "${MMACTION2_SRC}"

python - <<'PY'
from pathlib import Path
path = Path('/kaggle/working/mmaction2/mmaction/utils/dependency.py')
text = path.read_text()
old = "WITH_MULTIMODAL = all(\n    satisfy_requirement(item) for item in ['transformers>=4.28.0'])"
new = "# Disabled for this skeleton-only environment.\nWITH_MULTIMODAL = False"
if old in text:
    path.write_text(text.replace(old, new))
elif new not in text:
    raise RuntimeError(f'Could not disable MMAction2 multimodal imports in {path}')
PY

python - <<'PY'
from pathlib import Path
import mmaction
import mmaction.datasets
import mmaction.models

src = Path('/kaggle/working/mmaction2').resolve()
module = Path(mmaction.__file__).resolve()
assert module.is_relative_to(src), (module, src)
print('mmaction ->', mmaction.__version__, 'from', module)
PY


In [ ]:
# Restore the dataset and, if necessary, link an uploaded final workdir.
import glob, os, urllib.request

if not ANN_FILE.exists():
    hits = glob.glob('/kaggle/input/**/ntu60_2d.pkl', recursive=True)
    ANN_FILE.parent.mkdir(parents=True, exist_ok=True)
    if hits:
        ANN_FILE.symlink_to(Path(hits[0]).resolve())
    else:
        temporary = ANN_FILE.with_suffix('.pkl.part')
        temporary.unlink(missing_ok=True)
        urllib.request.urlretrieve(NTU60_URL, temporary)
        temporary.replace(ANN_FILE)
print('dataset:', ANN_FILE)

local_final = list(RESUME_WORK_DIR.glob('*epoch_16.pth'))
if not local_final:
    uploaded_final = sorted(Path('/kaggle/input').rglob('*epoch_16.pth'))
    uploaded_run_dirs = sorted({path.parent.resolve() for path in uploaded_final})
    if len(uploaded_run_dirs) != 1:
        raise FileNotFoundError(
            'No unique completed run directory found. Keep the completed '
            'resume workdir in this session or attach one Kaggle Model '
            'containing epoch_16.pth (regular and best copies may coexist). '
            f'Candidate directories: {uploaded_run_dirs}')
    uploaded_run_dir = uploaded_run_dirs[0]
    for source in uploaded_run_dir.rglob('*'):
        keep = (
            source.is_file() and (source.suffix in ('.pth', '.log', '.txt')
                                  or source.name in (
                                      'epoch_metrics.jsonl',
                                      'resume_state.json')))
        if keep:
            destination = RESUME_WORK_DIR / source.relative_to(uploaded_run_dir)
            destination.parent.mkdir(parents=True, exist_ok=True)
            if not destination.exists():
                destination.symlink_to(source.resolve())
    print('linked uploaded run:', uploaded_run_dir)

metric_sources = [
    path for directory in (ORIGINAL_WORK_DIR, RESUME_WORK_DIR)
    for path in directory.rglob('*')
    if path.is_file() and (path.name == 'epoch_metrics.jsonl'
                           or path.suffix in ('.log', '.txt'))]
if not metric_sources:
    raise FileNotFoundError(
        'Checkpoint found, but no epoch_metrics.jsonl/training_console.log '
        'was restored. Metrics are required to select the best epoch.')
print('metric sources:', *metric_sources, sep='\n- ')


In [ ]:
# Freeze the logged best checkpoint and independently infer all xsub_val samples.
import os, subprocess, sys

environment = os.environ.copy()
environment['PYTHONPATH'] = (
    str(MMACTION2_DIR) + os.pathsep + str(PROJECT_DIR) + os.pathsep
    + environment.get('PYTHONPATH', ''))
environment['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
environment['PYTHONUNBUFFERED'] = '1'
command = [
    sys.executable, 'scripts/evaluate_stgcn_joint.py',
    '--config', 'configs/stgcn_ntu60_xsub_80e_resume.py',
    '--ann-file', str(ANN_FILE),
    '--work-dir', str(ORIGINAL_WORK_DIR),
    '--work-dir', str(RESUME_WORK_DIR),
]
subprocess.run(command, cwd=PROJECT_DIR, env=environment, check=True)


In [ ]:
# Build every table, plot, report, and README-ready copy from raw predictions.
command = [
    sys.executable, 'scripts/analyze_stgcn_results.py',
    '--work-dir', str(ORIGINAL_WORK_DIR),
    '--work-dir', str(RESUME_WORK_DIR),
]
subprocess.run(command, cwd=PROJECT_DIR, env=environment, check=True)


In [ ]:
from IPython.display import Image, Markdown, display

summary_path = PROJECT_DIR / 'artifacts/analysis/experiment_summary.md'
display(Markdown(summary_path.read_text()))
display(Image(filename=str(PROJECT_DIR / 'artifacts/analysis/training_curve.png')))
display(Image(filename=str(PROJECT_DIR / 'artifacts/analysis/confusion_matrix_top20.png')))
print('All milestone artifacts:')
for path in sorted((PROJECT_DIR / 'artifacts').rglob('*')):
    if path.is_file():
        print('-', path.relative_to(PROJECT_DIR))
